# Project Policy Challenge -- Simple Qwen Starter

**Model:** `Qwen/Qwen2.5-0.5B-Instruct`  
**Task:** For each question, predict `YES`, `NO`, or `INSUFFICIENT_INFO` using frozen PSU policy passages.

This is deliberately a **naive baseline**, not a strong or sophisticated solution. It does just four things:

1. Find three policy snippets using TF-IDF.
2. Send them and the question to Qwen.
3. Read the first decision label from Qwen's response.
4. Calculate accuracy and macro-F1 on the **24 development questions**.

  
The policy passages are fixed, source-attributed **paraphrases** of official PSU pages, not full webpage copies. THESE ARE CLASSROOM RESEARCH MATERIALS, NOT OFFICIAL ADVISING GUIDANCE.

**For Kaggle:** Import this notebook; use **Add Input** to attach `PSU_Qwen_Simple_Starter_Data.zip`. Kaggle may unpack it into a dataset folder automatically. Enable Internet only if needed to install dependencies / download Qwen.


In [1]:
# 1. Check dependencies (no installation)

import torch
import transformers
import sklearn

from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Scikit-learn:", sklearn.__version__)

print("Dependencies loaded successfully!")

C:\Users\JOHND\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1+cpu
Transformers: 5.23.0
Scikit-learn: 1.9.1
Dependencies loaded successfully!


In [2]:
# 2. Load the frozen PSU excerpts, 24 questions, and development labels
from pathlib import Path
import csv, zipfile

filename = 'PSU_24_Questions_v1.csv'
roots = [Path('/kaggle/input'), Path.cwd(), Path('/kaggle/working')]
matches = [p for root in roots if root.exists() for p in root.rglob(filename)
           if (p.parent / 'PSU_24_Dev_Labels_v1.csv').exists()]

# Also works when Kaggle leaves the input ZIP unextracted.
if not matches:
    zips = [p for root in roots if root.exists()
            for p in root.rglob('PSU_Qwen_Simple_Starter_Data.zip')]
    if zips:
        destination = Path('/kaggle/working/psu_simple_data') if Path('/kaggle/working').exists() else Path.cwd()/'psu_simple_data'
        destination.mkdir(exist_ok=True, parents=True)
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(destination)
        matches = list(destination.rglob(filename))
assert matches, 'Attach PSU_Qwen_Simple_Starter_Data.zip using Kaggle Add Input.'
DATA_DIR = matches[0].parent

def read_csv(name):
    with (DATA_DIR/name).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

passages = read_csv('PSU_Policy_Excerpts_v1.csv')
questions = read_csv('PSU_24_Questions_v1.csv')
gold = {r['id']: r['gold'] for r in read_csv('PSU_24_Dev_Labels_v1.csv')}
print(f'{len(passages)} policy passages; {len(questions)} development questions')
print('Example:', questions[0]['question'])


23 policy passages; 24 development questions
Example: A PSU undergraduate has a term GPA of 2.30 and a PSU cumulative GPA of 2.10. Do both numbers satisfy the published GPA thresholds for good academic standing?


In [3]:
# 3. Very simple retrieval: return the top 3 passages by TF-IDF similarity
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')
docs = [p['heading'] + ' ' + p['text'] for p in passages]
doc_vectors = vectorizer.fit_transform(docs)
TOP_K = 3  # Teams can change this.

def retrieve(question):
    q_vector = vectorizer.transform([question])
    scores = (doc_vectors @ q_vector.T).toarray().ravel()
    indices = scores.argsort()[::-1][:TOP_K]
    return [passages[i] for i in indices]

example_evidence = retrieve(questions[0]['question'])
print('Retrieved IDs:', [p['excerpt_id'] for p in example_evidence])


Retrieved IDs: ['S01-E1', 'S01-E2', 'S02-E2']


In [4]:
# 4. Load Qwen and define a single naive prompting method
import torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu': torch.set_num_threads(4)
print('Loading on', DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
).to(DEVICE).eval()

PROMPT = '''Read the PSU policy excerpts and answer the question using ONLY the excerpts.
YES: The excerpts support the proposition in the question.
NO: The excerpts contradict it.
INSUFFICIENT_INFO: The excerpts do not establish either answer.
Do not treat missing evidence as NO.
Output only one label: YES, NO, or INSUFFICIENT_INFO.'''

def predict(question):
    evidence = retrieve(question)
    context = '\n'.join(f"[{p['excerpt_id']}] {p['text']}" for p in evidence)
    messages = [
        {'role': 'system', 'content': PROMPT},
        {'role': 'user', 'content': f'POLICY:\n{context}\n\nQUESTION: {question}'}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True, truncation=True, max_length=1800
    ).to(DEVICE)
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=20,
                                pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    match = re.match(r'^(?:(?:DECISION|ANSWER)\s*:\s*)?(YES|NO|INSUFFICIENT_INFO)\b', raw, re.I)
    label = match.group(1).upper() if match else 'INVALID'
    return label, raw, [p['excerpt_id'] for p in evidence]

print('Model ready')


Loading on cpu


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 661.12it/s]

Model ready


In [5]:

import numpy as np
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score
 
LABELS = ['YES', 'NO', 'INSUFFICIENT_INFO']
 
for l in LABELS:
    ids = tokenizer.encode(l, add_special_tokens=False)
    print(l, ids, [tokenizer.decode([i]) for i in ids])
 
first_ids = [tokenizer.encode(l, add_special_tokens=False)[0] for l in LABELS]
assert len(set(first_ids)) == 3, 'first tokens collide; rename the labels (e.g. A/B/C)'
 


YES [14004] ['YES']
NO [8996] ['NO']
INSUFFICIENT_INFO [9557, 8825, 89625, 9068] ['INS', 'UFF', 'ICIENT', '_INFO']


In [6]:
def label_scores(question):
    evidence = retrieve(question)
    context = '\n'.join(f"[{p['excerpt_id']}] {p['text']}" for p in evidence)
    messages = [
        {'role': 'system', 'content': PROMPT},
        {'role': 'user', 'content': f'POLICY:\n{context}\n\nQUESTION: {question}'},
    ]
    # No truncation here: the original truncation=True could cut the question
    # and generation prompt off the END of a long prompt. We check length instead.
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True,
    ).to(DEVICE)
    with torch.inference_mode():
        logits = model(**inputs).logits[0, -1].float()   # next-token logits
    logp = torch.log_softmax(logits, dim=-1)              # over the FULL vocab
    return {
        'logp': logp[first_ids].cpu().numpy(),            # log-prob of each label's first token
        'mass': float(logp[first_ids].exp().sum()),       # prob mass on the 3 labels
        'top_token': tokenizer.decode([int(logp.argmax())]),
        'n_tokens': inputs['input_ids'].shape[1],
        'ids': [p['excerpt_id'] for p in evidence],
    }
 
cache = {q['id']: label_scores(q['question']) for q in questions}
 
 

In [7]:
print('max prompt tokens:', max(c['n_tokens'] for c in cache.values()))
print('avg prob mass on the 3 labels:', round(np.mean([c['mass'] for c in cache.values()]), 3))
print('most common top tokens:', Counter(c['top_token'] for c in cache.values()).most_common(5))
# If mass is low or top tokens aren't YES/NO/INS..., the model wants to start with something
# else (e.g. "The"); fix the prompt before trusting any of the numbers below.
 

max prompt tokens: 294
avg prob mass on the 3 labels: 0.994
most common top tokens: [('NO', 20), ('YES', 4)]


In [8]:
def decide(logp, b_ins=0.0, b_no=0.0):
    s = logp.copy()
    s[1] += b_no
    s[2] += b_ins
    return LABELS[int(s.argmax())]
 
truth = [gold[q['id']] for q in questions]
print('bias  acc   macroF1  predictions')
for b in np.arange(0, 8.5, 0.5):
    preds = [decide(cache[q['id']]['logp'], b_ins=b) for q in questions]
    acc = accuracy_score(truth, preds)
    f1 = f1_score(truth, preds, labels=LABELS, average='macro', zero_division=0)
    print(f'{b:4.1f}  {acc:.3f}  {f1:.3f}   {dict(Counter(preds))}')
 


bias  acc   macroF1  predictions
 0.0  0.500  0.413   {'NO': 20, 'YES': 4}
 0.5  0.500  0.413   {'NO': 20, 'YES': 4}
 1.0  0.500  0.413   {'NO': 20, 'YES': 4}
 1.5  0.500  0.413   {'NO': 20, 'YES': 4}
 2.0  0.500  0.413   {'NO': 20, 'YES': 4}
 2.5  0.500  0.413   {'NO': 20, 'YES': 4}
 3.0  0.500  0.413   {'NO': 20, 'YES': 4}
 3.5  0.542  0.550   {'NO': 15, 'INSUFFICIENT_INFO': 5, 'YES': 4}
 4.0  0.375  0.378   {'INSUFFICIENT_INFO': 13, 'NO': 9, 'YES': 2}
 4.5  0.417  0.367   {'INSUFFICIENT_INFO': 19, 'YES': 2, 'NO': 3}
 5.0  0.417  0.311   {'INSUFFICIENT_INFO': 22, 'YES': 2}
 5.5  0.375  0.246   {'INSUFFICIENT_INFO': 23, 'YES': 1}
 6.0  0.333  0.167   {'INSUFFICIENT_INFO': 24}
 6.5  0.333  0.167   {'INSUFFICIENT_INFO': 24}
 7.0  0.333  0.167   {'INSUFFICIENT_INFO': 24}
 7.5  0.333  0.167   {'INSUFFICIENT_INFO': 24}
 8.0  0.333  0.167   {'INSUFFICIENT_INFO': 24}


In [9]:
for q in questions:
    if gold[q['id']] == 'INSUFFICIENT_INFO':
        s = cache[q['id']]['logp']
        p = np.exp(s - s.max()); p /= p.sum()             # softmax over just the 3 labels
        rank = int((s > s[2]).sum()) + 1                  # 1 = already winning
        gap = float(s.max() - s[2])                       # bias needed to make it win
        print(q['id'], f'rank={rank}', f'gap={gap:.2f}',
              dict(zip(LABELS, np.round(p, 2))))

def predict(question, b_ins=0.0):
    out = label_scores(question)
    label = decide(out['logp'], b_ins=b_ins)
    raw = 'logp(YES,NO,INS)=' + ','.join(f'{x:.2f}' for x in out['logp'])
    return label, raw, out['ids']

Q03 rank=3 gap=3.31 {'YES': np.float32(0.28), 'NO': np.float32(0.7), 'INSUFFICIENT_INFO': np.float32(0.03)}
Q06 rank=3 gap=4.08 {'YES': np.float32(0.18), 'NO': np.float32(0.81), 'INSUFFICIENT_INFO': np.float32(0.01)}
Q09 rank=3 gap=4.36 {'YES': np.float32(0.11), 'NO': np.float32(0.88), 'INSUFFICIENT_INFO': np.float32(0.01)}
Q12 rank=3 gap=4.28 {'YES': np.float32(0.38), 'NO': np.float32(0.61), 'INSUFFICIENT_INFO': np.float32(0.01)}
Q15 rank=3 gap=3.46 {'YES': np.float32(0.05), 'NO': np.float32(0.92), 'INSUFFICIENT_INFO': np.float32(0.03)}
Q18 rank=3 gap=3.92 {'YES': np.float32(0.15), 'NO': np.float32(0.84), 'INSUFFICIENT_INFO': np.float32(0.02)}
Q21 rank=3 gap=3.35 {'YES': np.float32(0.15), 'NO': np.float32(0.83), 'INSUFFICIENT_INFO': np.float32(0.03)}
Q24 rank=3 gap=4.58 {'YES': np.float32(0.05), 'NO': np.float32(0.94), 'INSUFFICIENT_INFO': np.float32(0.01)}


In [10]:
# 5. Run evaluation; start with 3, then change to 24
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
import pandas as pd

N_QUESTIONS = 24   # Change to 24 for the whole development set.
results = []
for q in questions[:N_QUESTIONS]:
    prediction, raw, ids = predict(q['question'])
    results.append({'id': q['id'], 'question': q['question'],
                    'gold': gold[q['id']], 'prediction': prediction,
                    'retrieved_ids': ';'.join(ids), 'raw_output': raw})
    print(q['id'], 'gold:', gold[q['id']], 'predicted:', prediction)

truth = [r['gold'] for r in results]
predictions = [r['prediction'] for r in results]
print('\nAccuracy:', round(accuracy_score(truth, predictions), 3))
print('Macro-F1:', round(f1_score(truth, predictions,
    labels=['YES', 'NO', 'INSUFFICIENT_INFO'], average='macro', zero_division=0), 3))
print('Predicted labels:', dict(Counter(predictions)))

# Compare with the deliberately trivial always-YES classifier.
print('Always-YES baseline:', round(sum(label == 'YES' for label in truth) / len(truth), 3))

out = Path('/kaggle/working/PSU_Qwen_Starter_Results.csv') if Path('/kaggle/working').exists() else Path('PSU_Qwen_Starter_Results.csv')
pd.DataFrame(results).to_csv(out, index=False)
print('Saved:', out)


Q01 gold: YES predicted: NO
Q02 gold: NO predicted: NO
Q03 gold: INSUFFICIENT_INFO predicted: NO
Q04 gold: YES predicted: NO
Q05 gold: NO predicted: NO
Q06 gold: INSUFFICIENT_INFO predicted: NO
Q07 gold: YES predicted: YES
Q08 gold: NO predicted: NO
Q09 gold: INSUFFICIENT_INFO predicted: NO
Q10 gold: YES predicted: NO
Q11 gold: NO predicted: NO
Q12 gold: INSUFFICIENT_INFO predicted: NO
Q13 gold: YES predicted: NO
Q14 gold: NO predicted: NO
Q15 gold: INSUFFICIENT_INFO predicted: NO
Q16 gold: YES predicted: YES
Q17 gold: NO predicted: NO
Q18 gold: INSUFFICIENT_INFO predicted: NO
Q19 gold: YES predicted: YES
Q20 gold: NO predicted: NO
Q21 gold: INSUFFICIENT_INFO predicted: NO
Q22 gold: YES predicted: YES
Q23 gold: NO predicted: NO
Q24 gold: INSUFFICIENT_INFO predicted: NO

Accuracy: 0.5
Macro-F1: 0.413
Predicted labels: {'NO': 20, 'YES': 4}
Always-YES baseline: 0.333
Saved: PSU_Qwen_Starter_Results.csv


## Starting points for teams

This simple baseline is intentionally imperfect. Teams may improve **retrieval**, **prompting**, **decoding**, **data preparation**, **fine-tuning**, or **evaluation**, but should keep the starting checkpoint fixed and compare on the same development questions.

**Important:** The gold labels are development-only and must never be inserted into a model prompt. The hidden test labels will remain private. Do not interpret these 24 examples as live advice about PSU policy.
